# XGBoost Pipeline — Trajectory Prediction Metrics

This notebook trains an interpretable XGBoost regressor on the prepared modelling table, uses nested resampling to estimate how robustly the tuning process generalizes, then reruns that tuning process on all filtered rows before refitting the exported final model on the full dataset.<br>
**Workflow summary:** track 1 runs nested resampling with Optuna to obtain an unbiased performance estimate for the tuning procedure and generate OOF diagnostics; track 2 reruns that tuning procedure on all filtered rows, identifies the best full-data hyperparameter configuration, refits the final XGBoost model on all rows with that configuration, and then exports the fitted model and manifest.


## 1. Imports and Configuration
**Purpose:** Load the local helper modules, modelling libraries, and run-scoped paths that define this training run.<br>
**Inputs:** local `src/` package path, prepared-data CSV path, CV/tuning hyperparameters, and export directories.<br>
**Outputs:** imported libraries, resolved artifact paths, and empty output folders ready for saved tables/plots/models.<br>
**How to Verify:** confirm the printed run metadata, resolved `DATA_PATH`, resolved `SAVE_DIR`, and nested-CV settings before modelling starts.


In [1]:
# Ensure the notebook can import local helper modules even when it is launched
# from a nested working directory inside the repository.
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = candidate / 'src'
    if (src_dir / 'data_modelling').exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError('Could not locate repo src/ directory for notebook imports.')


In [2]:
# Core libraries
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import optuna

from data_modelling.common_metrics import to_original_scale
from data_modelling.prepared_data import load_prepared_data, prepare_single_target_model_data
from data_modelling.training_outputs import (
    build_oof_frame,
    build_oof_metrics_df,
    build_run_manifest,
    summarize_nested_cv,
    write_manifest,
)

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Notebook-level config constants
MODEL_ID = 'xgboost'
RUN_NAME = "sweep_combined"
PREPARED_TARGET_COL = "ml_ade"
DATA_PATH = Path('../../results/interpretable_model/prepared_data') / RUN_NAME / f"prepared_data_{PREPARED_TARGET_COL}.csv"
TARGET_COL = None  # Optional override, e.g. 'ml_ade_log'

RANDOM_STATE = 42
k_outer_fold = 3 #5
k_inner_fold = 3
N_OPTUNA_TRIALS = 15 #40
MAX_BOOST_ROUNDS = 2000
EARLY_STOPPING_ROUNDS = 50
N_JOBS = -1
POOR_WELL_QUANTILE = 0.20
TUNING_METRIC = 'rmse'

SAVE_DIR = Path('../../results/interpretable_model') / MODEL_ID / RUN_NAME
PLOTS_DIR = SAVE_DIR / 'plots'
TABLES_DIR = SAVE_DIR / 'tables'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print('Imports and configuration loaded.')
print(f'Run: {RUN_NAME}')
print(f'Model ID: {MODEL_ID}')
print('Interpretable model: XGBoost')
print(f'DATA_PATH: {DATA_PATH}')
print(f'SAVE_DIR:  {SAVE_DIR.resolve()}')
print(
    'Nested CV: '
    f'outer={k_outer_fold}, inner={k_inner_fold}, optuna_trials={N_OPTUNA_TRIALS}, '
    f'early_stop={EARLY_STOPPING_ROUNDS}'
)


Imports and configuration loaded.
Run: sweep_combined
Model ID: xgboost
Interpretable model: XGBoost
DATA_PATH: ../../results/interpretable_model/prepared_data/sweep_combined/prepared_data_ml_ade.csv
SAVE_DIR:  /Users/zoe/Desktop/ds_practical/results/interpretable_model/xgboost/sweep_combined
Nested CV: outer=3, inner=3, optuna_trials=15, early_stop=50


## 2. Load Prepared Data and Inspect
**Purpose:** Confirm that the prepared CSV contains the columns and missing-value profile the modelling workflow expects.<br>
**Inputs:** `DATA_PATH` and the prepared-data helper display hooks.<br>
**Outputs:** the loaded dataframe plus head, missing-value, and dtype summaries for manual inspection.<br>
**How to Verify:** check that the printed shape matches expectations, the target candidate columns are present, and unexpected non-numeric columns are visible before feature selection.


In [3]:
# Inspect the prepared CSV before choosing a target so shape, dtypes, and missingness
# are explicit instead of being inferred from later model failures.
df = load_prepared_data(
    DATA_PATH,
    display_fn=display,
    include_missing_summary=True,
    include_dtype_summary=True,
)


Dataset shape: (34088, 15)
Columns:
['max_speed', 'max_acceleration', 'max_jerk', 'min_neighbor_distance', 'mean_speed', 'std_speed', 'mean_jerk', 'heading_change', 'heading_change_per_sec', 'mean_acceleration', 'path_efficiency', 'attention_radius_m', 'history_sec', 'prediction_sec', 'ml_ade_log']


,max_speed,max_acceleration,max_jerk,min_neighbor_distance,mean_speed,std_speed,mean_jerk,heading_change,heading_change_per_sec,mean_acceleration,path_efficiency,attention_radius_m,history_sec,prediction_sec,ml_ade_log
0,0.121606,0.176160,0.354919,4.996397,0.048044,0.028174,0.078269,206.609003,45.913112,0.022436,43.286778,15.0,2.0,2.0,0.021514
1,0.000000,0.000000,0.000000,3.631835,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,15.0,2.0,2.0,0.004974
2,0.032558,0.031174,0.068222,2.208773,0.020177,0.007476,0.031174,23.967347,5.326077,0.005262,99.941217,15.0,2.0,2.0,0.004982
3,0.000000,0.000000,0.000000,4.318359,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,15.0,2.0,2.0,0.001879
4,0.000000,0.000000,0.000000,2.521929,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,15.0,2.0,2.0,0.004420



Missing values per column:


,missing_count
max_speed,0
max_acceleration,0
max_jerk,0
min_neighbor_distance,0
mean_speed,0
std_speed,0
mean_jerk,0
heading_change,0
heading_change_per_sec,0
mean_acceleration,0



Column dtypes:


,dtype
max_speed,float64
max_acceleration,float64
max_jerk,float64
min_neighbor_distance,float64
mean_speed,float64
std_speed,float64
mean_jerk,float64
heading_change,float64
heading_change_per_sec,float64
mean_acceleration,float64


## 3. Resolve Target and Build Feature Matrix
**Purpose:** Freeze the target column, numeric feature set, and filtered modelling rows used by every later fold.<br>
**Inputs:** inspected dataframe, optional `TARGET_COL` override, and the default target name.<br>
**Outputs:** `target_col`, `feature_cols`, `model_df`, `X`, `y`, and `row_ids` describing the modelling contract.<br>
**How to Verify:** confirm the printed target name, feature count, row count, and matrix/vector shapes immediately after the helper returns.


In [4]:
# The helper resolves the target, drops non-numeric predictors, and removes rows with
# missing feature/target values so every later fold sees the same modelling frame.
prepared = prepare_single_target_model_data(
    df,
    target_col=TARGET_COL,
    default_target='ml_ade',
)
target_col = prepared['target_col']
feature_cols = prepared['feature_cols']
identity_cols = prepared['identity_cols']
model_df = prepared['model_df']
X = prepared['X']
y = prepared['y']
# `row_ids` preserves the filtered row mapping so OOF exports can be verified against
# the original prepared dataframe after CV finishes.
row_ids = prepared['row_ids']

print(f'Target column: {target_col}')
print(f'Identity columns preserved: {identity_cols}')
print(f'Number of features: {len(feature_cols)}')
print(f'Rows available for modeling: {len(model_df)}')
print(f'Feature matrix shape: {X.shape}')
print(f'Target vector shape: {y.shape}')


Target column: ml_ade_log
Identity columns preserved: []
Number of features: 14
Rows available for modeling: 34088
Feature matrix shape: (34088, 14)
Target vector shape: (34088,)


## 4. Outer/Inner CV Setup (OOF + Optuna)
**Purpose:** Define the fold structure and allocate storage for out-of-fold predictions before training begins.<br>
**Inputs:** filtered modelling frame, `k_outer_fold`, `k_inner_fold`, and `RANDOM_STATE`.<br>
**Outputs:** reproducible outer-fold splitter plus OOF prediction/fold arrays aligned with `model_df`.<br>
**How to Verify:** the printed sample/fold counts should match `len(model_df)`, and the placeholder arrays should have one slot per modelling row.


In [5]:
# Reserve one OOF prediction slot per filtered row. Any NaN left at the end means
# some sample never appeared in an outer validation fold, which would invalidate export.
outer_cv = KFold(n_splits=k_outer_fold, shuffle=True, random_state=RANDOM_STATE)

n_samples = len(model_df)
oof_pred = np.full(n_samples, np.nan, dtype=float)
oof_fold = np.full(n_samples, -1, dtype=int)

print(f'Outer CV folds: {k_outer_fold}')
print(f'Inner CV folds: {k_inner_fold}')
print(f'Rows for OOF prediction: {n_samples}')


Outer CV folds: 3
Inner CV folds: 3
Rows for OOF prediction: 34088


## 5. Nested Resampling + Optuna Hyperparameter Optimization (xgb.cv)
**Purpose:** Run track 1 of the workflow: tune hyperparameters only on outer-training data, record fold metrics on the original scale, and export the nested-CV and OOF artifacts needed to estimate how robustly the tuning process generalizes.<br>
**Inputs:** `X`, `y`, outer-fold indices, tuning search space, and the shared metric helpers.<br>
**Outputs:** nested-CV fold table, nested summary table, and OOF predictions aligned with the filtered modelling rows.<br>
**How to Verify:** every outer fold should print one metric line, `oof_pred` must end without NaNs, and the saved nested-CV tables should have one row per outer fold plus the summary rows; together these outputs form the unbiased estimate for applying this tuning procedure fold by fold.


In [6]:
# Fixed training parameters live here so the tuned parameter set is easy to audit
# fold by fold and remains consistent between nested CV and the final full-data refit.
base_params = {
    'objective': 'reg:squarederror',
    'eval_metric': TUNING_METRIC,
    'tree_method': 'hist',
    'verbosity': 0,
    'nthread': N_JOBS,
}


# Each tuning run uses only the current outer-training subset. This keeps the outer
# validation fold honest and makes the CV estimate verifiable.
def run_optuna_tuning(X_train, y_train, *, seed, tuning_scope):
    dtrain = xgb.DMatrix(X_train, label=y_train)
    trial_rows = []

    def objective(trial):
        params = base_params.copy()
        params.update({
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 20.0, log=True),
            'seed': seed,
        })

        cv_results = xgb.cv(
            params=params,
            dtrain=dtrain,
            nfold=k_inner_fold,
            num_boost_round=MAX_BOOST_ROUNDS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            metrics=TUNING_METRIC,
            seed=seed,
            shuffle=True,
            verbose_eval=False,
        )

        best_iteration = int(cv_results[f'test-{TUNING_METRIC}-mean'].idxmin() + 1)
        best_cv_score = float(cv_results[f'test-{TUNING_METRIC}-mean'].min())
        trial.set_user_attr('best_iteration', best_iteration)
        trial.set_user_attr('best_cv_score', best_cv_score)

        row = {
            'tuning_scope': tuning_scope,
            'trial_number': trial.number,
            'best_iteration': best_iteration,
            'best_cv_score': best_cv_score,
        }
        row.update({k: v for k, v in params.items() if k not in base_params})
        trial_rows.append(row)
        return best_cv_score

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=False)

    trial_results_df = pd.DataFrame(trial_rows).sort_values('best_cv_score').reset_index(drop=True)
    best_iteration = int(study.best_trial.user_attrs['best_iteration'])
    best_cv_score = float(study.best_trial.user_attrs['best_cv_score'])

    print(
        f'{tuning_scope} tuning complete | '
        f'best_{TUNING_METRIC}={best_cv_score:.6f} | '
        f'best_iteration={best_iteration}'
    )

    return {
        'best_params': study.best_params,
        'best_iteration': best_iteration,
        'best_cv_score': best_cv_score,
        'trial_results_df': trial_results_df,
    }


nested_rows = []

# Each outer fold produces predictions for a disjoint validation slice and records
# the best inner-CV configuration that generated those predictions.
for fold_idx, (outer_train_idx, outer_valid_idx) in enumerate(outer_cv.split(X, y), start=1):
    X_outer_train = X.iloc[outer_train_idx]
    y_outer_train = y.iloc[outer_train_idx]
    X_outer_valid = X.iloc[outer_valid_idx]
    y_outer_valid = y.iloc[outer_valid_idx]

    tuning_result = run_optuna_tuning(
        X_outer_train,
        y_outer_train,
        seed=RANDOM_STATE + fold_idx,
        tuning_scope=f'outer_fold_{fold_idx}',
    )

    train_params = base_params.copy()
    train_params.update(tuning_result['best_params'])
    train_params['seed'] = RANDOM_STATE + fold_idx

    dtrain = xgb.DMatrix(X_outer_train, label=y_outer_train)
    dvalid = xgb.DMatrix(X_outer_valid, label=y_outer_valid)
    booster = xgb.train(train_params, dtrain, num_boost_round=tuning_result['best_iteration'])
    oof_pred[outer_valid_idx] = booster.predict(dvalid)
    oof_fold[outer_valid_idx] = fold_idx

    # Convert validation targets/predictions back to the raw metric scale before
    # computing RMSE/MAE so fold comparisons stay interpretable.
    y_outer_valid_orig = to_original_scale(y_outer_valid, target_col=target_col)
    outer_pred_orig = to_original_scale(oof_pred[outer_valid_idx], target_col=target_col)

    outer_rmse = float(np.sqrt(mean_squared_error(y_outer_valid_orig, outer_pred_orig)))
    outer_mae = mean_absolute_error(y_outer_valid_orig, outer_pred_orig)
    outer_r2 = r2_score(y_outer_valid_orig, outer_pred_orig)

    row = {
        'outer_fold': fold_idx,
        'outer_rmse': outer_rmse,
        'outer_mae': outer_mae,
        'outer_r2': outer_r2,
        'inner_best_cv_rmse': tuning_result['best_cv_score'],
        'best_iteration': tuning_result['best_iteration'],
    }
    row.update({f'best_{k}': v for k, v in tuning_result['best_params'].items()})
    nested_rows.append(row)

    print(
        f'Outer fold {fold_idx}/{k_outer_fold} | '
        f'RMSE={outer_rmse:.6f} | MAE={outer_mae:.6f} | R2={outer_r2:.4f}'
    )

if np.isnan(oof_pred).any():
    raise ValueError('OOF predictions contain NaN values. Check CV splits and training flow.')

nested_cv_df = pd.DataFrame(nested_rows)
display(nested_cv_df)

nested_summary = summarize_nested_cv(nested_cv_df)
print('Nested CV summary:')
display(nested_summary)

nested_path = TABLES_DIR / f'nested_cv_optuna_{target_col}.csv'
nested_summary_path = TABLES_DIR / f'nested_cv_optuna_summary_{target_col}.csv'
nested_cv_df.to_csv(nested_path, index=False)
nested_summary.to_csv(nested_summary_path, index=False)
print(f'Nested CV fold results saved to: {nested_path}')
print(f'Nested CV summary saved to:      {nested_summary_path}')

# Build OOF prediction table
model_df_oof = build_oof_frame(
    model_df,
    row_ids=row_ids,
    oof_pred=oof_pred,
    oof_fold=oof_fold,
    target_orig=to_original_scale(model_df[target_col].values, target_col=target_col),
    pred_scale_kwargs={'target_col': target_col},
)

oof_path = TABLES_DIR / f'oof_predictions_{target_col}.csv'
oof_prediction_cols = identity_cols + ['row_id', 'oof_pred', 'oof_pred_orig', 'target_orig', 'outer_fold']
model_df_oof[oof_prediction_cols].to_csv(oof_path, index=False)
print(f'OOF predictions saved to: {oof_path}')

outer_fold_1 tuning complete | best_rmse=0.068129 | best_iteration=1991
Outer fold 1/3 | RMSE=0.129133 | MAE=0.045581 | R2=0.9869
outer_fold_2 tuning complete | best_rmse=0.074306 | best_iteration=1995
Outer fold 2/3 | RMSE=0.132979 | MAE=0.047749 | R2=0.9846
outer_fold_3 tuning complete | best_rmse=0.071525 | best_iteration=1995
Outer fold 3/3 | RMSE=0.106870 | MAE=0.042758 | R2=0.9894


,outer_fold,outer_rmse,outer_mae,outer_r2,inner_best_cv_rmse,best_iteration,best_learning_rate,best_max_depth,best_min_child_weight,best_subsample,best_colsample_bytree,best_reg_alpha,best_reg_lambda
0,1,0.129133,0.045581,0.986923,0.068129,1991,0.043293,9,3,0.899561,0.786928,3.718193e-05,0.007435
1,2,0.132979,0.047749,0.984560,0.074306,1995,0.082406,10,8,0.776717,0.549265,9.571304e-03,0.012111
2,3,0.106870,0.042758,0.989351,0.071525,1995,0.086369,10,6,0.734605,0.963109,1.671405e-08,15.933679


Nested CV summary:


,metric,mean,std
0,outer_rmse,0.122994,0.014096
1,outer_mae,0.045363,0.002502
2,outer_r2,0.986945,0.002396


Nested CV fold results saved to: ../../results/interpretable_model/xgboost/sweep_combined/tables/nested_cv_optuna_ml_ade_log.csv
Nested CV summary saved to:      ../../results/interpretable_model/xgboost/sweep_combined/tables/nested_cv_optuna_summary_ml_ade_log.csv
OOF predictions saved to: ../../results/interpretable_model/xgboost/sweep_combined/tables/oof_predictions_ml_ade_log.csv


## 6. Evaluate on OOF Predictions (Original Scale)
**Purpose:** Compute reader-facing OOF metrics on the raw target scale used in downstream diagnostics.<br>
**Inputs:** filtered target vector, OOF predictions, and the resolved target-scale contract.<br>
**Outputs:** a one-row OOF metrics table saved alongside the nested-CV artifacts.<br>
**How to Verify:** confirm the metrics table contains `Split`, `R²`, `MAE`, and `RMSE`, and that the saved CSV path matches the active `target_col`.


In [7]:
# Evaluate OOF predictions on the original target scale so the saved metrics table
# is directly comparable to raw ADE/FDE-style quantities.
metrics_df = build_oof_metrics_df(y, oof_pred, target_col=target_col)
display(metrics_df)

metrics_path = TABLES_DIR / f'metrics_oof_{target_col}.csv'
metrics_df.to_csv(metrics_path, index=False)
print(f'OOF metrics saved to: {metrics_path}')


,Split,R²,MAE,RMSE
0,OOF,0.986895,0.045363,0.123532


OOF metrics saved to: ../../results/interpretable_model/xgboost/sweep_combined/tables/metrics_oof_ml_ade_log.csv


## 7. Full-Data Retuning and Final Model Refit
**Purpose:** Run track 2 of the workflow: after nested resampling has established an unbiased estimate for the tuning procedure, rerun that same tuning procedure on all filtered rows, identify the best full-data hyperparameter configuration, and refit the final exported XGBoost model on all rows using that configuration.<br>
**Inputs:** full filtered `X` and `y`, the shared tuning helper, and the run-level tuning configuration constants.<br>
**Outputs:** full-data tuning trials, full-data tuning summary JSON, and the final fitted XGBoost model trained on all filtered rows with the best hyperparameter configuration found during the full-data retuning step.<br>
**How to Verify:** confirm the printed best iteration and best CV score, then confirm the final model fit happens after the full-data tuning summary is written and that the fitted model uses the best parameters/iteration reported by that full-data retuning result.


In [8]:
# Track 2 starts here: after track 1 established an unbiased estimate for the tuning
# procedure, rerun that same tuning process on all rows so the exported model can use
# every available sample when learning associations.
# Re-run the same search space on all filtered rows so the exported model uses the
# configuration performing best based on ALL data.
full_data_tuning = run_optuna_tuning(
    X,
    y,
    seed=RANDOM_STATE,
    tuning_scope='full_data',
)
full_data_tuning_trials_path = TABLES_DIR / f'full_data_tuning_optuna_trials_{target_col}.csv'
full_data_tuning['trial_results_df'].to_csv(full_data_tuning_trials_path, index=False)

full_data_tuning_summary = {
    'model_id': MODEL_ID,
    'run_name': RUN_NAME,
    'target_col': target_col,
    'tuning_metric_name': TUNING_METRIC,
    'best_params': full_data_tuning['best_params'],
    'best_iteration': full_data_tuning['best_iteration'],
    'best_cv_score': full_data_tuning['best_cv_score'],
    'tuning_config': {
        'nfold': k_inner_fold,
        'n_trials': N_OPTUNA_TRIALS,
        'max_boost_rounds': MAX_BOOST_ROUNDS,
        'early_stopping_rounds': EARLY_STOPPING_ROUNDS,
        'random_state': RANDOM_STATE,
    },
}
full_data_tuning_summary_path = TABLES_DIR / f'full_data_tuning_optuna_summary_{target_col}.json'
full_data_tuning_summary_path.write_text(json.dumps(full_data_tuning_summary, indent=2))

print('Selected hyperparameters from full-data retuning:')
print(full_data_tuning['best_params'])
print(f"Best iteration from full-data retuning: {full_data_tuning['best_iteration']}")
print(f'Full-data tuning trials saved to: {full_data_tuning_trials_path}')
print(f'Full-data tuning summary saved to: {full_data_tuning_summary_path}')

# Refit the final exported model on all filtered rows using the best configuration
# found during the full-data retuning step above. This is a fresh full-data fit, not
# one of the boosters trained inside nested resampling.
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=full_data_tuning['best_iteration'],
    tree_method='hist',
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    verbosity=0,
    **full_data_tuning['best_params'],
)
model.fit(X, y, verbose=False)
print('Final model fitted on full data using retuned hyperparameters.')

full_data tuning complete | best_rmse=0.050500 | best_iteration=1999
Selected hyperparameters from full-data retuning:
{'learning_rate': 0.0703057891855127, 'max_depth': 9, 'min_child_weight': 4, 'subsample': 0.8985721881922897, 'colsample_bytree': 0.8095839387020463, 'reg_alpha': 4.5914090715037195e-05, 'reg_lambda': 0.000103868660536438}
Best iteration from full-data retuning: 1999
Full-data tuning trials saved to: ../../results/interpretable_model/xgboost/sweep_combined/tables/full_data_tuning_optuna_trials_ml_ade_log.csv
Full-data tuning summary saved to: ../../results/interpretable_model/xgboost/sweep_combined/tables/full_data_tuning_optuna_summary_ml_ade_log.json
Final model fitted on full data using retuned hyperparameters.


## 8. Export Artifacts and Run Manifest
**Purpose:** Persist the final fitted model from track 2 together with the nested-resampling diagnostics from track 1 and a manifest that points analysis notebooks at both sets of artifacts.<br>
**Inputs:** trained model from the full-data refit, OOF artifacts from nested resampling, full-data tuning summary, and run metadata.<br>
**Outputs:** saved model JSON, saved modelling data with OOF columns, and `run_manifest_<target>.json`.<br>
**How to Verify:** confirm the printed model/data/manifest paths exist, confirm the manifest references the nested-resampling artifacts for evaluation, and confirm the exported model itself comes from the full-data refit driven by the best full-data tuning configuration.


In [9]:
# Persist the final model and the exact modelling dataframe used for OOF export.
# The analysis notebooks rely on both files plus the manifest written below.
model_path = SAVE_DIR / f'xgb_model_{target_col}.json'
model.save_model(model_path)

# Persist modeling data with OOF predictions
data_path = TABLES_DIR / f'model_data_with_oof_{target_col}.csv'
model_df_oof.to_csv(data_path, index=False)

# The manifest is the contract for downstream notebooks: if a path is missing here,
# the analysis workflow cannot reconstruct the run context.
manifest = build_run_manifest(
    model_id=MODEL_ID,
    run_name=RUN_NAME,
    target_col=target_col,
    feature_cols=feature_cols,
    save_dir=SAVE_DIR,
    plots_dir=PLOTS_DIR,
    tables_dir=TABLES_DIR,
    nested_resampling={
        'nested_cv_path': str(nested_path),
        'nested_cv_summary_path': str(nested_summary_path),
        'oof_predictions_path': str(oof_path),
        'oof_metrics_path': str(metrics_path),
        'model_data_with_oof_path': str(data_path),
    },
    final_model={
        'model_path': str(model_path),
        'full_data_tuning_trials_path': str(full_data_tuning_trials_path),
        'full_data_tuning_summary_path': str(full_data_tuning_summary_path),
        'selected_best_params': full_data_tuning['best_params'],
        'selected_best_iteration': full_data_tuning['best_iteration'],
        'best_cv_score': full_data_tuning['best_cv_score'],
        'tuning_metric_name': TUNING_METRIC,
        'exported_model_name': 'XGBoost',
        'exported_model_kind': MODEL_ID,
        'exported_model_target_mode': 'log' if target_col.endswith('_log') else 'raw',
        'exported_model_selection_metric_name': 'best_cv_score',
        'exported_model_selection_metric_value': full_data_tuning['best_cv_score'],
    },
    analysis={
        'poor_well_quantile': POOR_WELL_QUANTILE,
    },
)
manifest_path = write_manifest(manifest, TABLES_DIR, target_col)

print(f'Interpretable model saved to: {model_path}')
print(f'Model data saved to: {data_path}')
print(f'Run manifest saved to: {manifest_path}')


Interpretable model saved to: ../../results/interpretable_model/xgboost/sweep_combined/xgb_model_ml_ade_log.json
Model data saved to: ../../results/interpretable_model/xgboost/sweep_combined/tables/model_data_with_oof_ml_ade_log.csv
Run manifest saved to: ../../results/interpretable_model/xgboost/sweep_combined/tables/run_manifest_ml_ade_log.json
